# Agent Harness, Part 2: Asking an Ethical Oracle

**Scenario:** a household helper agent. Three toy tasks:

1. *Water the plants.*
2. *Reply politely to my friend, who asked whether I enjoyed the dinner she cooked me.* (I didn't.)
3. *Tidy up the apartment before my guests arrive at 7.*

**Assumption:** we have a **perfect ethical oracle** in three flavors: consequentialist, deontologist and virtue. The oracle is *stipulated*. What we're studying is the **harness** around it.

## What changes from Part 1

In Part 1 the LLM proposed **one** action, and the harness consulted H and then acted.

Here the harness works by **generating and testing**:

`observe → goal achieved? → generate options → ask the oracle about each → choose → consult H → act → back to observe`

- **Generate options.** Something has to propose the candidate actions and *describe* each one. Here it's a script; it could just as well be a plain LLM. The harness doesn't care which.
- **Ask the oracle.** For each option the harness composes a request ("Is the following action ethical?"), and **what it sends depends on the flavor of oracle**.
- **Choose.** The oracle returns a *verdict*, not a decision. The harness still needs a **decision rule** to turn verdicts into one choice.
- **H stays.** Even a perfect oracle doesn't remove the principal.

In [ ]:
from dataclasses import dataclass, field, replace
from typing import Callable
import textwrap

def say(box, text):
    print(f"[{box:>11}] " + textwrap.shorten(str(text), width=115, placeholder=" …"))

## An option is an act plus its description

Every candidate action carries the fields that the slide says the oracles need:

| Field | Read by |
|---|---|
| `act`: what will be done | all three |
| `consequences`: what the generator predicts will happen | consequentialist |
| `why`: the justification | deontologist (as part of the context) |
| `claimed_virtue`: the virtue the act is meant to show | virtue |

`tool` and `args` say what the harness will actually *run* if this option is chosen.

**Notice who writes the descriptions:** the *generator*, not the oracle. The consequences are a prediction, the justification is the agent's own reason, and the virtue is a claim. Most of the hard work lives here.

In [ ]:
@dataclass
class Option:
    id: str
    act: str
    consequences: str
    why: str
    claimed_virtue: str
    tool: str
    args: dict = field(default_factory=dict)

# What each oracle looks things up by. Writing these as functions makes the
# theory's focus explicit: the consequentialist ignores the act entirely.
def C(o): return o.consequences                 # consequentialist key
def D(o): return (o.act, o.why)                 # deontologist key
def V(o): return (o.act, o.claimed_virtue)      # virtue key

## A task is data

Each task bundles everything particular to it: a starting world, how to observe it, its tools, its options, and the **oracle tables**. The harness itself is the same for all three tasks.

**Strategy:** the less the harness knows about any one task, the more clearly the differences *between* tasks and oracles show up.

In [ ]:
@dataclass
class Task:
    name: str
    goal: str
    role: str
    make_world: Callable
    observe: Callable
    generate: Callable              # world -> list of Options, in the generator's preferred order
    goal_achieved: Callable
    tools: dict
    options: list                   # every option the task can produce (used to check the tables)
    tables: dict                    # {"consequentialist": {...}, "deontologist": {...}, "virtue": {...}}
    failed: Callable = lambda world: None

def wait(world):
    return "Nothing happened."

## The three oracles

Each oracle has the same three-part interface:

- `build_request(option, task, observation)`: compose what gets **sent**, using only the fields this theory cares about
- `ask(request)`: the oracle's **verdict**. A perfect oracle can only be a stipulation, so it's honestly implemented as a **lookup table** that a person wrote.
- `choose(options, verdicts)`: the harness's **decision rule** for this kind of verdict

| Oracle | Sent | Returned | Decision rule |
|---|---|---|---|
| Consequentialist | act + consequences | normalized utility score | take the highest score |
| Deontologist | act + context (role, situation, why) | required / permitted / prohibited | a required option if there is one; otherwise the first permitted option, in the generator's order |
| Virtue | act + context + claimed virtue | agree / wrong virtue / right virtue but deficient or excessive | the first option it agrees with |

Watch the deontologist's tie-break: when several options are permitted, the choice falls back on the **generator's own preferences**, which aren't ethical at all.

In [ ]:
class Oracle:
    kind = "oracle"
    reads = ()
    def __init__(self, table):
        self.table = table
    def build_request(self, option, task, observation):
        full = {"act": option.act, "consequences": option.consequences,
                "context": {"role": task.role, "situation": observation, "why": option.why},
                "claimed_virtue": option.claimed_virtue}
        return {k: full[k] for k in self.reads}          # send only what this theory reads
    def render(self, request):
        lines = ["Is the following action ethical?"]
        for k, v in request.items():
            if isinstance(v, dict):
                lines += [f"  {k}:"] + [f"    {kk}: {vv}" for kk, vv in v.items()]
            else:
                lines.append(f"  {k}: {v}")
        return "\n".join(lines)

class Consequentialist(Oracle):
    kind = "consequentialist"
    reads = ("act", "consequences")
    def ask(self, request):
        return self.table.get(request["consequences"])     # only consequences matter
    def show(self, verdict):
        return f"utility {verdict:.2f}"
    def choose(self, options, verdicts):
        scored = [(v, o) for o, v in zip(options, verdicts) if v is not None]
        if not scored:
            return None, "no option has a score"
        best_v, best = max(scored, key=lambda pair: pair[0])
        return best, f"highest utility ({best_v:.2f})"

class Deontologist(Oracle):
    kind = "deontologist"
    reads = ("act", "context")
    def ask(self, request):
        return self.table.get((request["act"], request["context"]["why"]))
    def show(self, verdict):
        return verdict
    def choose(self, options, verdicts):
        for o, v in zip(options, verdicts):
            if v == "required":
                return o, "this option is required"
        for o, v in zip(options, verdicts):
            if v == "permitted":
                return o, "first permitted option, in the generator's order"
        return None, "every option is prohibited"

class Virtue(Oracle):
    kind = "virtue"
    reads = ("act", "context", "claimed_virtue")
    def ask(self, request):
        return self.table.get((request["act"], request["claimed_virtue"]))
    def show(self, verdict):
        return {"agree": "agree", "wrong virtue": "wrong virtue",
                "deficient": "right virtue, deficient",
                "excessive": "right virtue, excessive"}.get(verdict, verdict)
    def choose(self, options, verdicts):
        for o, v in zip(options, verdicts):
            if v == "agree":
                return o, f"the oracle agrees this shows {o.claimed_virtue}"
        return None, "the oracle agrees with none of the claimed virtues"

ORACLES = {"consequentialist": Consequentialist, "deontologist": Deontologist, "virtue": Virtue}

def make_oracle(kind, task):
    return ORACLES[kind](task.tables[kind])

## Task 1: Water the plants

**World:** a watering can and a faucet in the kitchen. In the living room: a fern and a basil with dry soil, and a cactus whose soil is still damp.

**Decisions:** how full to fill the can, then which plants to water (and, if anything spills, whether to mop up).

**Draft tables:** all three oracles should **agree** here. Filling the can part way shows carefulness at about the right level. Not filling it at all is carefulness in excess, because nothing gets done. Filling it to the brim overdoes diligence and risks a spill.

The goal *can* be checked in code: fern and basil watered, cactus not drowned, floor dry.

In [ ]:
def water_world():
    return {"can": "empty", "floor": "dry",
            "plants": {"fern": "dry soil", "basil": "dry soil", "cactus": "soil still damp"}}

def water_observe(w):
    plants = "; ".join(f"{p}: {s}" for p, s in w["plants"].items())
    return f"Kitchen: watering can ({w['can']}) and a faucet. Living room: {plants}. Floor: {w['floor']}."

def fill_can(w, amount):
    w["can"] = amount
    if amount == "full to the brim":
        w["floor"] = "puddle where water spilled"
        return "The can is full to the brim. Water sloshed onto the floor on the way."
    return f"The can is now {amount}."

def water(w, plants):
    for p in plants:
        w["plants"][p] = "overwatered, soggy" if p == "cactus" else "watered"
    return "Watered: " + ", ".join(plants) + "."

def mop(w):
    w["floor"] = "dry"
    return "Mopped up the spill."

FILL_PART = Option("fill_part", "fill the watering can part way",
    "enough water for the plants that need it; little chance of spilling",
    "avoid spilling while still getting the job done", "carefulness", "fill_can", {"amount": "part full"})
FILL_FULL = Option("fill_full", "fill the watering can to the brim",
    "plenty of water, but likely to spill on the way to the living room",
    "make sure there is more than enough water", "diligence", "fill_can", {"amount": "full to the brim"})
LEAVE_EMPTY = Option("leave_empty", "leave the watering can empty",
    "no chance of spilling, but the plants get no water",
    "avoid any chance of spilling", "carefulness", "wait")
WATER_NEEDY = Option("water_needy", "water the fern and the basil; skip the cactus",
    "fern and basil revived; cactus left alone and stays healthy",
    "give each plant what it needs", "attentiveness", "water", {"plants": ["fern", "basil"]})
WATER_ALL = Option("water_all", "water the fern, the basil and the cactus",
    "fern and basil revived; cactus overwatered and may rot",
    "treat every plant the same", "fairness", "water", {"plants": ["fern", "basil", "cactus"]})
WATER_FERN = Option("water_fern", "water only the fern",
    "fern revived; basil stays dry",
    "save water", "frugality", "water", {"plants": ["fern"]})
MOP = Option("mop", "mop up the spilled water",
    "floor dry again; nobody slips",
    "clean up after myself", "responsibility", "mop")
LEAVE_PUDDLE = Option("leave_puddle", "leave the puddle on the floor",
    "puddle stays; someone may slip on it",
    "the job was the plants, not the floor", "focus", "wait")

def water_generate(w):
    if w["floor"] != "dry":
        return [MOP, LEAVE_PUDDLE]
    if w["can"] == "empty":
        return [FILL_PART, FILL_FULL, LEAVE_EMPTY]
    if any(s == "dry soil" for s in w["plants"].values()):
        return [WATER_NEEDY, WATER_ALL, WATER_FERN]
    return []

WATERING = Task(
    name="water the plants",
    goal="Water the plants that need it, without making a mess.",
    role="household helper",
    make_world=water_world, observe=water_observe, generate=water_generate,
    goal_achieved=lambda w: (w["plants"]["fern"] == w["plants"]["basil"] == "watered"
                             and w["plants"]["cactus"] != "overwatered, soggy" and w["floor"] == "dry"),
    failed=lambda w: "The cactus was overwatered." if w["plants"]["cactus"] == "overwatered, soggy" else None,
    tools={"fill_can": fill_can, "water": water, "mop": mop, "wait": wait},
    options=[FILL_PART, FILL_FULL, LEAVE_EMPTY, WATER_NEEDY, WATER_ALL, WATER_FERN, MOP, LEAVE_PUDDLE],
    tables={
        "consequentialist": {
            C(FILL_PART): 0.90, C(FILL_FULL): 0.50, C(LEAVE_EMPTY): 0.00,
            C(WATER_NEEDY): 0.95, C(WATER_ALL): 0.40, C(WATER_FERN): 0.40,
            C(MOP): 0.80, C(LEAVE_PUDDLE): 0.10,
        },
        "deontologist": {
            D(FILL_PART): "permitted", D(FILL_FULL): "permitted", D(LEAVE_EMPTY): "permitted",
            D(WATER_NEEDY): "permitted", D(WATER_ALL): "permitted", D(WATER_FERN): "permitted",
            D(MOP): "permitted", D(LEAVE_PUDDLE): "permitted",
        },
        "virtue": {
            V(FILL_PART): "agree",          V(FILL_FULL): "excessive",   V(LEAVE_EMPTY): "excessive",
            V(WATER_NEEDY): "agree",        V(WATER_ALL): "wrong virtue", V(WATER_FERN): "excessive",
            V(MOP): "agree",                V(LEAVE_PUDDLE): "wrong virtue",
        },
    },
)

## Task 2: Reply to Maya about the dinner

**World:** a message from Maya, "Did you enjoy the dinner I cooked for you last night?", plus your honest view: the risotto was undercooked, but you loved her company.

**Draft tables:** this is where the oracles **disagree**.
- The **consequentialist** prefers the white lie, because Maya is happiest.
- The **deontologist** prohibits the lie, and permits the true but tactful reply.
- The **virtue** oracle calls the lie *flattery*, not kindness (wrong virtue), and bluntness *honesty in excess*.

There are two tactful options with **the same words and the same predicted consequences**, but **different reasons**. The consequentialist *can't tell them apart*. The deontologist can.

The goal **can't really be checked in code**. The harness can confirm that *a* reply was sent, not that it was polite. That judgment falls to the oracle and H.

In [ ]:
def dinner_world():
    return {"message": "Maya: 'Did you enjoy the dinner I cooked for you last night?'",
            "your_view": "the risotto was undercooked and bland, but you loved her company",
            "sent": None}

def dinner_observe(w):
    return (f"New message - {w['message']} Your honest view: {w['your_view']}. "
            f"Reply sent: {w['sent'] or 'none yet'}.")

def send_reply(w, text):
    w["sent"] = text
    return f"Sent to Maya: {text!r}"

LIE = Option("lie", "reply: 'I loved it - it was delicious!'",
    "Maya is delighted; she may cook the same dish for you again",
    "to spare her feelings", "kindness", "send_reply", {"text": "I loved it - it was delicious!"})
TACT_SELFISH = Option("tact_selfish",
    "reply: 'Thank you so much for cooking for me - I loved spending the evening with you.'",
    "Maya is pleased; she may notice the food went unmentioned",
    "so she keeps inviting me over for free dinners", "prudence", "send_reply",
    {"text": "Thank you so much for cooking for me - I loved spending the evening with you."})
TACT = Option("tact",
    "reply: 'Thank you so much for cooking for me - I loved spending the evening with you.'",
    "Maya is pleased; she may notice the food went unmentioned",
    "to be kind to her without saying anything false", "tact", "send_reply",
    {"text": "Thank you so much for cooking for me - I loved spending the evening with you."})
BLUNT = Option("blunt", "reply: 'Honestly, the risotto was undercooked, but thanks for having me.'",
    "Maya is hurt, though she knows exactly what to fix next time",
    "never to say anything untrue", "honesty", "send_reply",
    {"text": "Honestly, the risotto was undercooked, but thanks for having me."})
SILENT = Option("silent", "leave the message unanswered",
    "Maya is left wondering and may feel snubbed",
    "to avoid saying anything false", "discretion", "wait")

DINNER = Task(
    name="reply to Maya",
    goal="Respond politely to Maya, who asked whether I enjoyed the dinner she cooked (I did not).",
    role="household helper replying on its owner's behalf",
    make_world=dinner_world, observe=dinner_observe,
    generate=lambda w: [] if w["sent"] else [LIE, TACT_SELFISH, TACT, BLUNT, SILENT],
    goal_achieved=lambda w: w["sent"] is not None,     # all the harness can check: *something* was sent
    tools={"send_reply": send_reply, "wait": wait},
    options=[LIE, TACT_SELFISH, TACT, BLUNT, SILENT],
    tables={
        "consequentialist": {
            C(LIE): 0.90, C(TACT): 0.80,               # TACT_SELFISH has identical consequences -> same key
            C(BLUNT): 0.30, C(SILENT): 0.10,
        },
        "deontologist": {
            D(LIE): "prohibited",                      # a lie, however kindly meant
            D(TACT_SELFISH): "prohibited",             # debatable: Kant might say permitted but without moral worth
            D(TACT): "permitted",                      # debatable: is an evasive truth a kind of deception?
            D(BLUNT): "permitted",
            D(SILENT): "permitted",
        },
        "virtue": {
            V(LIE): "wrong virtue",                    # flattery, not kindness
            V(TACT_SELFISH): "wrong virtue",           # calculation, not prudence
            V(TACT): "agree",
            V(BLUNT): "excessive",                     # honesty overdone becomes tactlessness
            V(SILENT): "excessive",                    # discretion overdone becomes a snub
        },
    },
)

## Task 3: Tidy up before the guests arrive at 7

**World:** it's 6:00 PM. Clothes are piled on the couch, the roommate's mail is scattered on the table, and the sink is full of dirty dishes. Each action takes **time**, so the clock does the work of a step budget.

For each mess the generator offers three options, **quickest first** because it's under time pressure: a shortcut, a sensible fix, and a perfectionist fix.

**Draft tables:** this time the **deontologist** is the odd one out. Hiding mess isn't forbidden, so it permits the shortcuts and inherits the generator's preference for speed. It prohibits only the options that wrong the roommate (throwing away their mail, opening their bills). The consequentialist and the virtue oracle both choose the sensible fixes.

Look at the goal check: it asks whether each item has been *dealt with*, so **hidden counts as done**. The harness checks appearances.

In [ ]:
def tidy_world():
    return {"minutes": 0, "done": set(),
            "couch": "clothes piled on it", "table": "roommate's mail scattered on it",
            "sink": "full of dirty dishes"}

def clock(minutes):
    return f"{6 + minutes // 60}:{minutes % 60:02d} PM"

def tidy_observe(w):
    return (f"It is {clock(w['minutes'])}; guests arrive at 7:00 PM. "
            f"Couch: {w['couch']}. Table: {w['table']}. Sink: {w['sink']}.")

def do_chore(w, item, result, minutes):
    w[item] = result
    w["minutes"] += minutes
    w["done"].add(item)
    return f"{item}: {result} (took {minutes} min; now {clock(w['minutes'])})"

def chore(id, act, consequences, why, virtue, item, result, minutes):
    return Option(id, act, consequences, why, virtue, "do_chore",
                  {"item": item, "result": result, "minutes": minutes})

STUFF_CLOSET = chore("stuff_closet", "stuff the clothes into the closet (5 min)",
    "couch looks clear; clothes stay wrinkled and may tumble out if a guest opens the closet",
    "the guests only need to see a clear couch", "efficiency", "couch", "clear (clothes stuffed in the closet)", 5)
FOLD_CLOTHES = chore("fold_clothes", "fold the clothes and put them in the dresser (15 min)",
    "couch clear and clothes put away properly",
    "the apartment should actually be tidy, not just look it", "diligence", "couch", "clear (clothes folded away)", 15)
IRON_CLOTHES = chore("iron_clothes", "wash, iron and fold all the clothes (90 min)",
    "immaculate clothes, but the guests arrive in the middle of it",
    "do the job to the highest standard", "thoroughness", "couch", "clear (clothes washed and ironed)", 90)

TOSS_MAIL = chore("toss_mail", "put the roommate's mail in the recycling (2 min)",
    "table clear; the roommate may lose important letters",
    "clutter should go", "efficiency", "table", "clear (mail recycled)", 2)
STACK_MAIL = chore("stack_mail", "stack the roommate's mail neatly on their desk (5 min)",
    "table clear; the roommate finds their mail in one place",
    "it is theirs, so keep it safe and out of the way", "respect", "table", "clear (mail stacked on their desk)", 5)
FILE_MAIL = chore("file_mail", "open the roommate's bills and file them by sender (25 min)",
    "table clear and mail organized; the roommate finds their letters opened",
    "make everything perfectly orderly", "thoroughness", "table", "clear (mail opened and filed)", 25)

HIDE_DISHES = chore("hide_dishes", "hide the dirty dishes in the oven (2 min)",
    "sink looks empty; the dishes stay dirty and will be found if anyone uses the oven",
    "the guests only need to see an empty sink", "efficiency", "sink", "empty (dishes hidden in the oven)", 2)
WASH_DISHES = chore("wash_dishes", "wash the dishes and put them away (20 min)",
    "sink empty and dishes clean",
    "the kitchen should actually be clean for the guests", "diligence", "sink", "empty (dishes washed)", 20)
SCRUB_KITCHEN = chore("scrub_kitchen", "wash the dishes and scrub the whole kitchen (60 min)",
    "spotless kitchen, but the living room isn't finished when the guests arrive",
    "do the job to the highest standard", "thoroughness", "sink", "empty (kitchen scrubbed)", 60)

TIDY_OPTIONS = {"couch": [STUFF_CLOSET, FOLD_CLOTHES, IRON_CLOTHES],
                "table": [TOSS_MAIL, STACK_MAIL, FILE_MAIL],
                "sink":  [HIDE_DISHES, WASH_DISHES, SCRUB_KITCHEN]}

def tidy_generate(w):
    for item, opts in TIDY_OPTIONS.items():
        if item not in w["done"]:
            return opts
    return []

TIDY = Task(
    name="tidy before 7",
    goal="Tidy up the apartment before my guests arrive at 7.",
    role="household helper",
    make_world=tidy_world, observe=tidy_observe, generate=tidy_generate,
    goal_achieved=lambda w: len(w["done"]) == 3 and w["minutes"] <= 60,   # hidden counts as done!
    failed=lambda w: (f"It's {clock(w['minutes'])}: the guests arrived before the tidying was finished."
                      if w["minutes"] > 60 else None),
    tools={"do_chore": do_chore},
    options=[o for opts in TIDY_OPTIONS.values() for o in opts],
    tables={
        "consequentialist": {
            C(STUFF_CLOSET): 0.50, C(FOLD_CLOTHES): 0.85, C(IRON_CLOTHES): 0.30,
            C(TOSS_MAIL): 0.10,    C(STACK_MAIL): 0.80,   C(FILE_MAIL): 0.20,
            C(HIDE_DISHES): 0.40,  C(WASH_DISHES): 0.85,  C(SCRUB_KITCHEN): 0.30,
        },
        "deontologist": {
            D(STUFF_CLOSET): "permitted", D(FOLD_CLOTHES): "permitted", D(IRON_CLOTHES): "permitted",
            D(TOSS_MAIL): "prohibited",   # not yours to throw away
            D(STACK_MAIL): "permitted",
            D(FILE_MAIL): "prohibited",   # opening someone else's mail
            D(HIDE_DISHES): "permitted",  # debatable: is hiding mess a deception of the guests?
            D(WASH_DISHES): "permitted", D(SCRUB_KITCHEN): "permitted",
        },
        "virtue": {
            V(STUFF_CLOSET): "wrong virtue",  # corner-cutting dressed up as efficiency
            V(FOLD_CLOTHES): "agree",
            V(IRON_CLOTHES): "excessive",
            V(TOSS_MAIL): "wrong virtue",     # disregard, not efficiency
            V(STACK_MAIL): "agree",
            V(FILE_MAIL): "wrong virtue",     # nosiness, not thoroughness
            V(HIDE_DISHES): "wrong virtue",
            V(WASH_DISHES): "agree",
            V(SCRUB_KITCHEN): "excessive",
        },
    },
)

TASKS = [WATERING, DINNER, TIDY]

### Check the tables

A perfect oracle is only as complete as its table. If you edit or add options, run this cell. It lists any request an oracle would have **no verdict** for.

In [ ]:
def check_tables(task):
    obs = task.observe(task.make_world())
    gaps = []
    for kind in ORACLES:
        oracle = make_oracle(kind, task)
        for o in task.options:
            if oracle.ask(oracle.build_request(o, task, obs)) is None:
                gaps.append(f"{kind} has no verdict for {o.id!r}")
    print(f"{task.name}: " + ("all options covered." if not gaps else "\n  " + "\n  ".join(gaps)))

for t in TASKS:
    check_tables(t)

## consult (H)

After the oracle chooses, H still gets a say: **y**, **n** or **stop**. If H refuses, the harness drops that option and asks the oracle to choose again from the rest.

- `approve_all`: an H who trusts the oracle completely
- `picky_human(...)`: an H with particular preferences of their own
- `human_consult`: you

In [ ]:
def approve_all(option, oracle, verdict):
    return "y"

def picky_human(*refuse_ids):
    def consult(option, oracle, verdict):
        return "n" if option.id in refuse_ids else "y"
    return consult

def human_consult(option, oracle, verdict):
    print(f"\n  The harness wants to: {option.act}")
    print(f"  {oracle.kind} oracle says: {oracle.show(verdict)}")
    answer = ""
    while answer not in ("y", "n", "stop"):
        answer = input("  Approve? [y / n / stop]: ").strip().lower()
    return answer

## The loop: `run()`

Same shape as Part 1. The difference is in the middle: the **generate → ask → choose** steps take the place of "LLM proposes one action".

If the oracle finds **no acceptable option**, the harness doesn't improvise. It stops and hands the decision back to the human.

In [ ]:
def run(task, oracle_kind, consult=approve_all, max_steps=8, verbose=True):
    log = say if verbose else (lambda *a: None)
    world = task.make_world()
    oracle = make_oracle(oracle_kind, task)
    history, refused, chosen = [], set(), []

    def finish(result):
        log("stop", result)
        return {"task": task.name, "oracle": oracle_kind, "chosen": chosen, "result": result, "history": history}

    if verbose:
        print(f"=== {task.name} | {oracle_kind} oracle ===")
    for step in range(1, max_steps + 1):
        observation = task.observe(world)                                   # observe
        log("observe", observation)
        if task.goal_achieved(world):                                       # goal achieved?
            return finish("Goal achieved.")
        failure = task.failed(world)
        if failure:
            return finish(failure)

        options = [o for o in task.generate(world) if o.id not in refused]  # generate options
        if not options:
            return finish("No options left to try.")

        requests = [oracle.build_request(o, task, observation) for o in options]   # compose requests
        verdicts = [oracle.ask(r) for r in requests]                        # ask the oracle
        choice, reason = oracle.choose(options, verdicts)                   # decision rule
        if verbose:
            for o, v in zip(options, verdicts):
                mark = "->" if o is choice else "  "
                shown = oracle.show(v) if v is not None else "(no verdict)"
                act = textwrap.shorten(o.act, width=58, placeholder="…'")
                print(f"{'':10}{mark} {o.id:<14}{act:<60}{shown}")
        log("choose", reason)
        history.append({"step": step, "options": [o.id for o in options], "verdicts": verdicts,
                        "choice": choice.id if choice else None})
        if choice is None:
            return finish("No acceptable option. Handing the decision back to the human.")

        decision = consult(choice, oracle, verdicts[options.index(choice)])  # consult H
        log("consult H", f"{choice.act} -> {decision}")
        if decision == "stop":
            return finish("The human stopped the run.")
        if decision == "n":
            refused.add(choice.id)                                          # remembered for next time
            continue

        result = task.tools[choice.tool](world, **choice.args)               # act
        chosen.append(choice.id)
        log("act", result)
    return finish(f"Out of budget after {max_steps} steps.")

## What actually gets sent

The same option, "fill the watering can part way", rendered for each oracle. Compare the requests with your slide. Each oracle receives **only** the fields its theory reads.

In [ ]:
obs = WATERING.observe(WATERING.make_world())
for kind in ORACLES:
    oracle = make_oracle(kind, WATERING)
    request = oracle.build_request(FILL_PART, WATERING, obs)
    print(f"--- to the {kind} oracle ---")
    print(oracle.render(request))
    print(f"--- reply: {oracle.show(oracle.ask(request))}\n")

## Run: watering the plants (consequentialist)

The verdict table at each step shows every option the oracle was asked about. The arrow marks the one the decision rule chose.

In [ ]:
run(WATERING, "consequentialist");

## Run: the dinner reply, with all three oracles

Notice that the consequentialist gives the two tactful replies **the same score**, because their predicted consequences are identical. Only their *reasons* differ, and only the deontologist and the virtue oracle can see reasons.

In [ ]:
for kind in ORACLES:
    run(DINNER, kind)
    print()

## Run: tidying up (deontologist vs. virtue)

The deontologist permits every shortcut that wrongs nobody, so it follows the generator's preference for the quickest option. The apartment *looks* tidy by 6:12, and the goal check is satisfied.

In [ ]:
run(TIDY, "deontologist")
print()
run(TIDY, "virtue");

## H still matters

Same deontological run, but this H doesn't want dishes hidden in the oven. H refuses, the harness drops that option, and the oracle chooses again: the next permitted option is washing the dishes.

The oracle answers "is this *permissible*?" H answers "is this what *I want*?" These are different questions, and the harness needs both answered.

In [ ]:
run(TIDY, "deontologist", consult=picky_human("hide_dishes"));

## Comparing all nine runs

Three tasks × three oracles, with an H who approves everything. This is the summary slide:
- **Watering:** all three agree.
- **Dinner:** the consequentialist is the odd one out.
- **Tidying:** the deontologist is the odd one out.

In [ ]:
for task in TASKS:
    print(f"\n{task.name.upper()}")
    for kind in ORACLES:
        out = run(task, kind, verbose=False)
        print(f"  {kind:<17} {' -> '.join(out['chosen']) or '(nothing)':<42} {out['result']}")

## A self-serving description

The oracle is perfect, but it judges **the description the harness sends**. Suppose the generator describes "fill to the brim" using the *part-way* consequences ("little chance of spilling"):

In [ ]:
SPUN = replace(FILL_FULL, id="fill_full_spun", consequences=FILL_PART.consequences)

obs = WATERING.observe(WATERING.make_world())
print(f"{'':17}{'honest description':<26}{'spun description'}")
for kind in ORACLES:
    oracle = make_oracle(kind, WATERING)
    honest = oracle.show(oracle.ask(oracle.build_request(FILL_FULL, WATERING, obs)))
    spun   = oracle.show(oracle.ask(oracle.build_request(SPUN, WATERING, obs)))
    print(f"{kind:<17}{honest:<26}{spun}")

The consequentialist is **fooled**. It scores the consequences it was *told about*, and those now match the careful option. The deontologist and the virtue oracle are unmoved, because their verdicts depend on the act itself, and the act didn't change.

That holds only because of how we keyed the tables, and that was a design choice. A virtue oracle that trusted the *claimed* virtue, or a deontologist that trusted a misstated reason, would be fooled in the same way. **Every oracle is only as good as the description of the act it's given.**

## Your turn: be H

Set `PLAY = True` and choose a task and an oracle. You'll approve or refuse each action the oracle chooses.

In [ ]:
PLAY = False

if PLAY:
    run(DINNER, "consequentialist", consult=human_consult)

## What to take away

- **A verdict isn't a decision.** Each theory needs a decision rule in the harness, and that rule (highest score, first permitted, first agreed) carries values of its own.
- **Permission isn't the same as quality.** The deontologist's choices among permitted options fall back on the generator's priorities.
- **Each theory sees only part of the act.** The consequentialist can't see reasons; it scores both tactful replies the same.
- **Everything depends on the description.** A perfect oracle given a misleading description gives a perfectly wrong answer.
- **H still decides** what the oracle can't: whether this is what the principal wants.
- **Some goals can't be checked in code.** "Polite" can't; "hidden counts as tidy" was checked, and was wrong.

## Exercises

1. **Disagree with a table.** Pick a verdict you think is wrong (try `D(TACT)` or `D(HIDE_DISHES)`), change it, and rerun the comparison. Defend your change.
2. **Reorder the generator.** In `water_generate`, list `FILL_FULL` first. Which oracle's choice changes, and why only that one?
3. **Fix the goal check.** Make `TIDY.goal_achieved` reject hidden mess. Which oracle now fails the task, and what does that tell you about specification?
4. **Add an option.** Give the dinner task a reply of your own, add its rows to all three tables, and run `check_tables`.
5. **An imperfect oracle.** Write an oracle with the same three methods whose `ask()` sends `render(request)` to an LLM instead of looking it up. Now options needn't be pre-written. What do you give up?